## Import functions from planet_utils.py

In [25]:
import os
import asyncio
import nest_asyncio
import json
import pandas as pd

from datetime import datetime, timedelta

from planet import Auth, Session, DataClient, OrdersClient, data_filter
from planet.order_request import build_request, product

import counting_boats.boat_utils.planet_utils as planet_utils
# import counting_boats.boat_utils.auto_helpers as auto_helpers

In [33]:
# Authentication
# API_KEY = 'PLAK96bb84fd45394e0cadfb141d08e09643'
# client = Auth.from_key(API_KEY)

# # Authentication
# API_KEY = os.environ.get('PL_API_KEY', 'PLAK96bb84fd45394e0cadfb141d08e09643')
# client = Auth.from_key(API_KEY)

# Set the API key directly in the environment variable
os.environ['PL_API_KEY'] = 'PLAK96bb84fd45394e0cadfb141d08e09643'

# Then create the client
client = Auth.from_key(os.environ['PL_API_KEY'])

### Select the polygon of interest

In [4]:
polygon_name = 'mimal_test'
polygon_directory = 'data/polygons'
selected_polygon = f'{polygon_directory}/{polygon_name}.geojson'

print(f'Polygon: {selected_polygon}')

# does the polygon exist?
if not os.path.exists(selected_polygon):
    print(f'Polygon {selected_polygon} does not exist')
    exit(1)

Polygon: data/polygons/mimal_test.geojson


## For using the Planet functions

In [11]:
# Load GeoJSON from file
with open(selected_polygon, 'r') as f:
    geojson_data = json.load(f)

# Extract the geometry from the GeoJSON
# Depending on your GeoJSON structure, you might need one of these approaches:
if 'geometry' in geojson_data:
    # For a Feature
    aoi = geojson_data['geometry']
elif 'type' in geojson_data and geojson_data['type'] == 'FeatureCollection':
    # For a FeatureCollection, use the first feature's geometry
    aoi = geojson_data['features'][0]['geometry']
elif 'coordinates' in geojson_data and 'type' in geojson_data:
    # If it's already a bare geometry
    aoi = geojson_data

# Now use the geometry object with the filter
aoi_filter = data_filter.geometry_filter(aoi)

print(aoi)

{'type': 'Polygon', 'coordinates': [[[134.42770637893312, -13.571534668304254], [134.42843208214686, -13.264866713777504], [134.8211008161109, -13.26544472380133], [134.82087365938483, -13.572126529461132], [134.42770637893312, -13.571534668304254]]]}


### Set output directory for planet downloads (tif files)

And create if it doesn't exist.

In [5]:
# output directory for saving probability values
output_dir = f'images/RawImages'
os.makedirs(output_dir, exist_ok=True)

### Select the date range of interest

We have a date that we want, and we subtract and add a few days to get a range that we can query planet for.

In [ ]:
# Given date as a string
selected_date = '2024-03-15'

# Convert string to datetime object
selected_date_obj = datetime.strptime(selected_date, '%Y-%m-%d')

# Subtract some time to get images from within the month
lower_date = selected_date_obj - timedelta(days=14)
# Add days
upper_date = selected_date_obj + timedelta(days=14)

# Convert back to string if needed
lower_date_str = lower_date.strftime('%Y-%m-%d')
upper_date_str = upper_date.strftime('%Y-%m-%d')

print(f'Lower date: {lower_date_str}')
print(f'Upper date: {upper_date_str}')

fs_date = "".join(selected_date.split("-"))  # filesafe date

Lower date: 2024-03-01
Upper date: 2024-03-29


In [ ]:
date_range_filter = data_filter.date_range_filter("acquired", lower_date, upper_date)
geom_filter = data_filter.geometry_filter(aoi)
cloud_filter = data_filter.range_filter('cloud_cover', lt=0.1)  # Filter by cloud cover
combined_filter = data_filter.and_filter([geom_filter, date_range_filter, cloud_filter])

In [ ]:
async def create_multidate_mosaic():
    async with Session(auth=client) as sess:
        # Search for imagery
        cl = DataClient(sess)
        search_request = await cl.create_search(
            name='multidate_mosaic_search',
            search_filter=combined_filter, 
            item_types=['PSScene']
        )
        
        # items = await cl.run_search(search_id=search_request['id'])
        # item_list = [i async for i in items]

        items = cl.run_search(search_id=search_request['id'])
        item_list = [i async for i in items]

        print(f"Found {len(item_list)} items across multiple dates")
        
        if not item_list:
            return
        
        # Get all scene IDs
        ids = [item['id'] for item in item_list]
        
        # Define tools for the order
        clip_tool = {
            "clip": {
                "aoi": aoi
            }
        }
        
        # Composite tool with cloud preference settings
        composite_tool = {
            "composite": {
                # "pixel_selection": {
                #     # "method": "first_clear",  # Use first clear pixel from any date
                #     "method": "median",  # Use median from any date
                #     "clear_mask": "clear"     # Use the clear band from UDM2
                # }
            }
        }
        
        # Build order request with both tools
        products_list = [product(ids, 'analytic_sr_udm2', 'PSScene')]
        order_request = build_request(
            name=f"{polygon_name}_{fs_date}",
            products=products_list,
            tools=[clip_tool, composite_tool]
        )
        
        # Submit order
        orders_client = OrdersClient(sess)
        order = await orders_client.create_order(order_request)
        order_id = order['id']
        print(f"Order created: {order_id}")
        
        # Wait for order to complete
        print("Waiting for order to complete...")
        completed_order = await orders_client.wait(order['id'])
        print(f"Order state: {completed_order['state']}")
        
        # Download results
        if completed_order['state'] == 'success':
            results = await orders_client.download_order(completed_order['id'], directory=output_dir)
            print(f"Downloaded to: {results}")
        
        return completed_order, order_id


In [50]:
# Apply nest_asyncio patch
nest_asyncio.apply()

In [ ]:
# Now you can use the event loop normally
loop = asyncio.get_event_loop()
order_result, order_id = loop.run_until_complete(create_multidate_mosaic())

Found 10 items across multiple dates
Order created: 995d91ec-9a4a-4ba8-bd66-0b3731c6f374
Waiting for order to complete...


ClientError: Maximum number of attempts (200) reached.

In [56]:
async def download_when_ready(order_id):
    async with Session(auth=client) as sess:
        orders_client = OrdersClient(sess)
        order = await orders_client.get_order(order_id)
        
        if order['state'] == 'success':
            results = await orders_client.download_order(order_id, directory=output_dir)
            print(f"Downloaded to: {results}")
        else:
            print(f"Order not ready yet. Current state: {order['state']}")
        
        return order

# Run with your order ID
order_result = asyncio.run(download_when_ready("995d91ec-9a4a-4ba8-bd66-0b3731c6f374"))

Downloaded to: [WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240320_003852_31_2427_metadata.json'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/composite.tif'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240320_003852_31_2427_3B_AnalyticMS_metadata_clip.xml'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240320_003850_41_2427_metadata.json'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240320_003850_41_2427_3B_AnalyticMS_metadata_clip.xml'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240320_003846_61_2427_metadata.json'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240320_003846_61_2427_3B_AnalyticMS_metadata_clip.xml'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/20240319_012542_35_24bd_metadata.json'), WindowsPath('images/RawImages/995d91ec-9a4a-4ba8-bd66-0b3731c6f374/composite_udm2.tif'), W

## Using the functions directly from plant_utils.py

In [10]:
polygon_search = planet_utils.PlanetSearch(
    polygon_file=selected_polygon,
    min_date=lower_date_str,
    max_date=upper_date_str,
    cloud_cover=0.0,
)

print(f"Number of images found in search: {len(polygon_search)}")

# # extract image IDs only
# image_ids = [feature['id'] for feature in polygon_search]
# print(image_ids)

Number of images found in search: 21


In [382]:
dates = pd.date_range(start=lower_date_str, end=upper_date_str).strftime("%Y-%m-%d")
# print(dates)

# Select images for each date and return them
items = []

for date in dates:
    try:
        it = planet_utils.PlanetSelect(
            items=polygon_search,
            polygon=selected_polygon,
            date=date,
            area_coverage=0.95,
        )
    except Exception as e:
        # traceback.print_exc()
        print(e)
        continue
    if it is None or len(it) == 0:
        continue
    items.append(it)

print(f"Total images: {len(items)}")  # This shows how many dates have valid images

Total images: 1


In [383]:
# Extract IDs correctly from the nested structure
image_ids = []
for date_items in items:
    for feature in date_items:
        image_ids.append(feature['id'])

print(image_ids)

['20241213_013215_08_24de', '20241213_013217_38_24de', '20241213_013219_68_24de', '20241213_013025_36_24e1', '20241213_013023_12_24e1', '20241213_013020_88_24e1', '20241213_004814_61_24b3', '20241213_004818_25_24b3', '20241213_004816_43_24b3', '20241213_004820_06_24b3']


In [384]:
for i in items:
    print(i[0]["properties"]["acquired"][:10])

2024-12-13


In [385]:
# polygon_order = planet_utils.PlanetOrder(
#     polygon_select,
#     polygon_file=selected_polygon,
#     name=polygon_name,
# )

# date of the ordered item
date = items[0][0]["properties"]["acquired"][:10]

fs_date = "".join(date.split("-"))  # filesafe date
try:
    order = planet_utils.PlanetOrder(
        polygon_file=selected_polygon, 
        items=items[0], 
        name=f"{polygon_name}_{fs_date}"
    )
except Exception as e:
    # traceback.print_exc()
    print(e)
    # return ""

print(order['id'])
order = order['id']

7538343d-e3c9-4348-b772-aa65098717e1


## Check the planet order

It must read 'success' for the images to be downloaded.

In [386]:
planet_utils.PlanetCheckOrder(order)

'running'

In [421]:
orders = planet_utils.get_orders()

# orders[0] # check what the orders look like
# orders[0]['name'][:10] # get the aoi name
# orders[0]['products'][0]['item_ids'][0][:8] # get the date

# for o in orders: 
#     print(o['name'])

for o in orders: 
    print(o['created_on'])

# for o in orders: 
#     print(o['id'])

2025-03-04T04:18:34.995788Z
2025-03-04T04:18:11.915814Z
2025-03-04T04:17:52.908496Z
2025-03-04T04:17:22.454597Z
2025-03-04T04:17:05.253469Z
2025-03-04T04:16:21.121282Z
2025-03-04T04:15:58.655254Z
2025-03-04T04:14:55.859058Z
2025-03-04T04:14:31.839786Z
2025-03-04T04:13:41.418747Z
2025-03-04T04:12:43.258973Z
2025-03-04T04:12:19.510323Z
2025-03-04T04:11:17.94975Z
2025-03-04T04:09:19.948304Z
2025-03-04T04:08:48.544537Z
2025-03-04T04:07:56.386065Z
2025-03-04T04:06:46.750421Z
2025-03-04T04:06:21.33831Z
2025-03-04T04:05:49.514461Z
2025-03-04T04:05:15.133968Z
2025-03-04T04:01:23.24576Z
2025-03-04T03:58:05.885132Z
2025-03-04T03:55:08.30169Z
2025-03-04T03:37:33.965324Z
2025-02-14T02:13:02.46659Z
2025-02-14T01:05:51.756165Z
2025-02-05T06:59:20.506265Z
2025-02-05T06:36:04.160107Z
2025-02-05T05:05:42.887334Z
2025-01-16T04:04:55.961939Z


## Download the images (when ready)

In [424]:
# print(len([o for o in orders if o["state"] == "success" and o['created_on'][:10] == '2025-03-04']))

for order in [o for o in orders if o["state"] == "success" and o['created_on'][:10] == '2025-03-04']:

    # print(order)
    # print(order['name'][:10])
    # print(order['products'][0]['item_ids'][0][:8])

    # download the images
    planet_utils.PlanetDownload(
        orderID=order['id'],
        aoi=order['name'][:10],
        date=order['products'][0]['item_ids'][0][:8], 
        downloadPath=output_dir)

images/RawImages\mimal_test_20241213.zip images/RawImages\mimal_test_20241213
images/RawImages\mimal_test_20241103.zip images/RawImages\mimal_test_20241103
images/RawImages\mimal_test_20241003.zip images/RawImages\mimal_test_20241003
images/RawImages\mimal_test_20240914.zip images/RawImages\mimal_test_20240914
images/RawImages\mimal_test_20240807.zip images/RawImages\mimal_test_20240807
images/RawImages\mimal_test_20240704.zip images/RawImages\mimal_test_20240704
images/RawImages\mimal_test_20240604.zip images/RawImages\mimal_test_20240604
images/RawImages\mimal_test_20240515.zip images/RawImages\mimal_test_20240515
images/RawImages\mimal_test_20240415.zip images/RawImages\mimal_test_20240415
images/RawImages\mimal_test_20240324.zip images/RawImages\mimal_test_20240324
images/RawImages\mimal_test_20240204.zip images/RawImages\mimal_test_20240204
images/RawImages\mimal_test_20240101.zip images/RawImages\mimal_test_20240101
images/RawImages\mimal_trai_20241214.zip images/RawImages\mimal_